# components/preview_panel

> Collapsible preview panel with audio/video player and file metadata

In [ ]:
#| default_exp components.preview_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Optional

from fasthtml.common import Div, Span, Input, P

from cjm_fasthtml_media_gallery.components.players import render_audio_player, render_video_player

from cjm_fasthtml_daisyui.components.data_display.collapse import (
    collapse, collapse_title, collapse_content, collapse_modifiers
)
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, max_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, truncate
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import divide
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V10 P2 content_panel + I1 inset divider)
from cjm_fasthtml_design_system.panels import panels
from cjm_fasthtml_design_system.insets import insets

from cjm_transcription_source_select.models import SelectedFile
from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds
from cjm_transcription_source_select.utils import format_file_size, format_duration

## File Metadata Display

Shows path, format, size, and duration for the previewed file.

In [ ]:
#| export
def _render_metadata_row(
    label: str,  # Label text
    value: str,  # Value text
) -> Div:  # Metadata row element
    """Render a single metadata label-value pair."""
    return Div(
        Span(label, cls=combine_classes(font_size.xs, text_dui.base_content.opacity(60), w(16))),
        Span(value, cls=combine_classes(font_size.xs, truncate)),
        cls=combine_classes(str(flex_display), items.center, gap(2))
    )

In [ ]:
#| export
def _render_file_metadata(
    selected_file: SelectedFile,  # File to display metadata for
) -> Div:  # Metadata section
    """Render file metadata section."""
    path = selected_file.get("path", "")
    fmt = selected_file.get("format", "")
    size_bytes = selected_file.get("size_bytes", 0)
    duration = selected_file.get("duration")

    rows = [
        _render_metadata_row("Path", path),
        _render_metadata_row("Format", fmt.upper() if fmt else "Unknown"),
        _render_metadata_row("Size", format_file_size(size_bytes) if size_bytes else "Unknown"),
    ]
    if duration is not None:
        rows.append(_render_metadata_row("Duration", format_duration(duration)))

    return Div(
        *rows,
        cls=combine_classes(
            str(flex_display), flex_direction.col, gap(1),
            p(3), insets.divider_30, border_radius.box
        )
    )

## Preview Panel

Collapsible panel that displays the media player and file metadata. Uses DaisyUI's collapse component with a checkbox to control open/close state.

In [ ]:
#| export
def render_preview_panel(
    selected_file: Optional[SelectedFile] = None,  # File to preview (None for empty state)
    media_src_url: str = "",  # Base URL for media file serving
    is_open: bool = False,  # Whether panel starts open
) -> Div:  # Preview panel component
    """Render a collapsible preview panel with audio/video player."""
    # Empty state
    if selected_file is None:
        return Div(
            Input(
                type="checkbox",
                id=SourceSelectHtmlIds.PREVIEW_TOGGLE,
            ),
            Div(
                lucide_icon("eye", size=4, cls=str(text_dui.base_content.opacity(60))),
                Span("Preview", cls=combine_classes(font_weight.bold, font_size.sm, m.l(2))),
                cls=combine_classes(collapse_title, str(flex_display), items.center)
            ),
            Div(
                P(
                    "Click a file in the selection to preview it",
                    cls=combine_classes(
                        text_dui.base_content.opacity(40), font_size.sm, p(4)
                    )
                ),
                id=SourceSelectHtmlIds.PREVIEW_CONTENT,
                cls=combine_classes(collapse_content)
            ),
            id=SourceSelectHtmlIds.PREVIEW_PANEL,
            cls=combine_classes(
                collapse, collapse_modifiers.arrow,
                w.full, panels.content_panel, m.t(4)
            )
        )

    # Active state with player
    path = selected_file["path"]
    filename = selected_file.get("filename", path.rsplit("/", 1)[-1])
    file_type = selected_file.get("file_type", "audio")

    # Build media URL with path query parameter
    file_url = f"{media_src_url}?path={path}" if media_src_url else ""

    # Render the appropriate player
    if file_type == "video":
        player = render_video_player(file_url, cls=str(w.full))
    else:
        player = render_audio_player(file_url)

    # Type badge
    if file_type == "video":
        type_badge = Span("video", cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm, text_dui.secondary))
    else:
        type_badge = Span("audio", cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm, text_dui.info))

    return Div(
        Input(
            type="checkbox",
            checked="checked" if is_open else None,
            id=SourceSelectHtmlIds.PREVIEW_TOGGLE,
        ),
        # Collapse title
        Div(
            lucide_icon("eye", size=4, cls=str(text_dui.base_content.opacity(60))),
            Span("Preview", cls=combine_classes(font_weight.bold, font_size.sm, m.l(2))),
            Span(filename, cls=combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.l(2), truncate)),
            type_badge,
            cls=combine_classes(collapse_title, str(flex_display), items.center, gap(2))
        ),
        # Collapse content
        Div(
            Div(
                player,
                cls=combine_classes(p(3))
            ),
            _render_file_metadata(selected_file),
            id=SourceSelectHtmlIds.PREVIEW_CONTENT,
            cls=combine_classes(collapse_content, p(4))
        ),
        id=SourceSelectHtmlIds.PREVIEW_PANEL,
        cls=combine_classes(
            collapse, collapse_modifiers.arrow,
            w.full, panels.content_panel, m.t(4)
        )
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()